In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

from unsloth import FastVisionModel
from datasets import load_dataset
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
from transformers import TextStreamer

MODEL_NAME = "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit"
OUTPUT_DIR = "outputs"
SEED = 3407
MAX_STEPS = 50

INSTRUCTION = """
You are a world-class OCR expert specializing in recognizing all types of vehicle license plates
(cars, motorbikes, trucks, etc.) in any weather or lighting condition, including blurred, dirty,
or low-contrast images. Your recognition must be precise and avoid any confusion between
similar-looking characters (e.g., '0' and 'O', '1' and 'I', '8' and 'B').

Analyze the given image, which may contain one or multiple license plates.
For each license plate detected, extract and return ONLY its exact content,
using only the following valid characters: digits (0-9), uppercase letters (A-Z),
the hyphen (-), and the dot (.).

List each license plate you find on a separate line, with no extra words,
symbols, or explanations.
"""

model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth"
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=SEED,
)

dataset = load_dataset("EZCon/taiwan-license-plate-recognition", split="train")
dataset = dataset.remove_columns(["xywhr", "is_electric_car"])
dataset = dataset.rename_column("license_number", "text")

def convert_to_conversation(sample):
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": INSTRUCTION},
                    {"type": "image", "image": sample["image"]}
                ]
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": sample["text"]}
                ]
            }
        ]
    }

converted_dataset = [convert_to_conversation(sample) for sample in dataset]

FastVisionModel.for_inference(model)

test_image = dataset[2]["image"]
messages = [
    {"role": "user", "content": [
        {"type": "image", "image": test_image},
        {"type": "text", "text": INSTRUCTION}
    ]}
]

input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(test_image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")

print("\nBefore training:")
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
result = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=converted_dataset,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=MAX_STEPS,
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=SEED,
        output_dir=OUTPUT_DIR,
        report_to="none",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=2048,
    ),
)

trainer_stats = trainer.train()

FastVisionModel.for_inference(model)

print("\nAfter training:")
result = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

model.save_pretrained("Models/unsloth_finetune")
tokenizer.save_pretrained("Models/unsloth_finetune")


  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-sc8tzals/unsloth_932114af876647199dfab7dfcdba72ab
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-sc8tzals/unsloth_932114af876647199dfab7dfcdba72ab
  Resolved https://github.com/unslothai/unsloth.git to commit eeb49d54b8d801a1ce922fa15f778e2bad205db0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.6/869.6 kB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 20.8 MB/s eta 0:00:00


model.safetensors:   0%|          | 0.00/1.54G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/572 [00:00<?, ?B/s]

...

Step,Training Loss
1,2.578605
2,2.568652
3,2.553040
4,2.480093
5,2.338987
6,2.153503
7,1.917072
8,1.646196
9,1.407159
10,1.189265


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-50/tokenizer_config.json.
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



After training:
MYK-8008<|im_end|>


Unsloth: Restored added_tokens_decoder metadata in Models/unsloth_finetune/tokenizer_config.json.


['Models/unsloth_finetune/processor_config.json']

In [4]:
!mv /content/Models/unsloth_finetune /content/drive/MyDrive/VLM-FineTuned/Models/


mv: inter-device move failed: '/content/Models/unsloth_finetune' to '/content/drive/MyDrive/VLM-FineTuned/Models/unsloth_finetune'; unable to remove target: Directory not empty


In [ ]:
!streamlit run main.py & sleep 5 && ssh -o StrictHostKeyChecking=no -R 80:localhost:8501 nokey@localhost.run




2026-05-25 09:25:16.555 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.252.161.11:8501


Welcome to localhost.run!

Follow your favourite reverse tunnel at [https://twitter.com/localhost_run].

To set up and manage custom domains go to https://admin.localhost.run/

More details on custom domains (and how to enable subdomains of your custom
domain) at https://localhost.run/docs/custom-domains

If you get a permission denied error check the faq for how to connect with a key or
create a free tunnel without a key at [http://localhost:3000/docs/faq#generating-an-ssh-key].

To explore using localhost.run visit the documentation site:
https://localhost.run/docs/


** your connection id is 7c19d8e9-bb93-4036-91ea-1911a149c69e, please mention it if you send me a message about an issue. **

authenticated as anonymous user
a852f959d06688.lhr.life t